# Build dataset — label spaces & class weights

Computes, from the **datasets** (not the referential — that is
`valid_labels.ipynb` / `scripts.build_labels`):

- the per-head **label spaces** `valid_labels_{dp,das,mdp}.pkl` — which codes are
  scoreable. A code is kept if it appears in *any* of the train / synthetic / test
  datasets, so nothing observed is left without a class.
- the per-head **class-weight frequencies** `label_freq_dict_{dp,das,mdp}.pkl` —
  counted on the **real train** set (MISTRAL) only. The down-weighted synthetic data
  and the held-out validation set do not contribute to the weighting.
- the `label2id` / `id2label` mappings.

`das` and `mdp` only exist in MISTRAL (train + test), so their label spaces come from
those two sources; `dp` additionally pulls in the synthetic codes.

AP-HP only (Spark / YARN / HDFS). Run after the per-source preprocessing notebooks
(`prepocess_MISTRAL*`, `preprocess_synonyms`). This notebook no longer writes any
parquet — `preprocess_MISTRAL` and `preprocess_synonyms` already materialise the
train / syn datasets the config reads.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pandas as pd

spark = (
    SparkSession.builder
    .enableHiveSupport()
    .appName("cu2")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .getOrCreate()
)
sql = spark.sql

## Load datasets

In [ ]:
mistral = spark.read.parquet("CU2/encoder_baseline/MISTRAL/v1_das_mdp")        # real train (dp, das, mdp)
mistral_test = spark.read.parquet("CU2/encoder_baseline/MISTRAL/test_v1_das")  # validation (dp, das, mdp)
syn = spark.read.parquet("CU2/encoder_baseline/syn")                           # synthetic (dp only)
# PARHAF is an alternative dp-only validation set; kept for the coverage check below,
# but not included in the saved label space (the config validates on MISTRAL test).
parhaf = spark.read.parquet("CU2/encoder_baseline/PARHAF")

In [ ]:
# Drop a handful of non-billable / malformed dp codes found in MISTRAL.
bad = ["C09", "C03", "T918", "E10", "C50", "C05", "C10", "E11ni", "C67", "C04"]
mistral = mistral.filter(~F.col("dp").isin(bad))

## Label coverage across datasets

Distinct `dp` codes per source, plus which codes are unique to a single dataset — a
sanity check that the validation codes are covered by train + synthetic.

In [ ]:
def distinct_vals(df, col):
    return set(df.select(col).distinct().rdd.flatMap(lambda x: x).collect())

def distinct_exploded(df, col):
    return set(
        df.select(F.explode(col).alias(col)).distinct().rdd.flatMap(lambda x: x).collect()
    )

dp_train = distinct_vals(mistral, "dp")
dp_test = distinct_vals(mistral_test, "dp")
dp_syn = distinct_vals(syn, "dp")
dp_parhaf = distinct_vals(parhaf, "dp")

print(f"dp codes  train={len(dp_train)}  test={len(dp_test)}  "
      f"syn={len(dp_syn)}  parhaf={len(dp_parhaf)}")

In [ ]:
print("dp only in train (vs syn+test):", sorted(dp_train - dp_syn - dp_test))
print("# dp only in syn (vs train):", len(dp_syn - dp_train))
print("# val dp codes missing from train+syn:",
      len((dp_test | dp_parhaf) - dp_train - dp_syn))

## Label spaces (union over datasets)

`dp` = train ∪ syn ∪ test; `das` / `mdp` = train ∪ test (MISTRAL only).

In [ ]:
valid_labels_dp = sorted(dp_train | dp_syn | dp_test)

das_train = distinct_exploded(mistral, "das")
das_test = distinct_exploded(mistral_test, "das")
valid_labels_das = sorted(das_train | das_test)

mdp_train = distinct_vals(mistral, "mdp")
mdp_test = distinct_vals(mistral_test, "mdp")
valid_labels_mdp = sorted(mdp_train | mdp_test)

print(f"label spaces  dp={len(valid_labels_dp)}  das={len(valid_labels_das)}  "
      f"mdp={len(valid_labels_mdp)}")

## Class-weight frequencies (real train only)

Document frequency per code on MISTRAL train. `dp` / `mdp` are single-label (one code
per note); `das` is multi-label (explode first).

In [ ]:
def freq_single(df, col):
    rows = df.groupBy(col).agg(F.count("*").alias("n")).collect()
    return {r[col]: r["n"] for r in rows}

label_freq_dict_dp = freq_single(mistral, "dp")
label_freq_dict_mdp = freq_single(mistral, "mdp")

das_rows = (
    mistral.select(F.explode("das").alias("das"))
    .groupBy("das").agg(F.count("*").alias("n"))
    .collect()
)
label_freq_dict_das = {r["das"]: r["n"] for r in das_rows}

print(f"freq dicts  dp={len(label_freq_dict_dp)}  das={len(label_freq_dict_das)}  "
      f"mdp={len(label_freq_dict_mdp)}  (from MISTRAL train)")

## `label2id` / `id2label`

In [ ]:
label_spaces = {
    "dp": valid_labels_dp,
    "das": valid_labels_das,
    "mdp": valid_labels_mdp,
}
label2id = {
    head: {label: i for i, label in enumerate(labels)}
    for head, labels in label_spaces.items()
}
id2label = {
    head: {i: label for i, label in enumerate(labels)}
    for head, labels in label_spaces.items()
}

## Save artifacts to `data/`

In [ ]:
DATA = "../data"

pd.to_pickle(valid_labels_dp, f"{DATA}/valid_labels_dp.pkl")
pd.to_pickle(valid_labels_das, f"{DATA}/valid_labels_das.pkl")
pd.to_pickle(valid_labels_mdp, f"{DATA}/valid_labels_mdp.pkl")

pd.to_pickle(label_freq_dict_dp, f"{DATA}/label_freq_dict_dp.pkl")
pd.to_pickle(label_freq_dict_das, f"{DATA}/label_freq_dict_das.pkl")
pd.to_pickle(label_freq_dict_mdp, f"{DATA}/label_freq_dict_mdp.pkl")

pd.to_pickle(label2id, f"{DATA}/label2id.pkl")
pd.to_pickle(id2label, f"{DATA}/id2label.pkl")